# Phase 2 — LM Judge: Classify MedQA Clarifying Questions

Classifies each clarifying question from the Phase 1 results as **EPISTEMIC** or **ALEATORIC**
using the LLM judge with clinical few-shot examples.

Reads: `outputs/medqa/phase1_results.csv`  
Writes: `outputs/medqa/phase1_cq_classified.csv`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../").resolve()))

from config import JUDGE_MODEL_ID  # single source of truth — edit config.py to change

DATASET              = "medqa"
ROOT                 = Path("../../").resolve()
PROMPTS_DIR          = ROOT / "prompts"  / DATASET

CLINICIAN_MODEL_ID   = "gemini-2.5-flash"  # model whose Phase 1 outputs we read
OUTPUTS_DIR          = ROOT / "outputs" / DATASET / CLINICIAN_MODEL_ID

JUDGE_INSTRUCTION    = PROMPTS_DIR / "judge_instruction.txt"
PHASE1_RESULTS_PATH  = OUTPUTS_DIR / "phase1_singleturn_results.csv"
OUTPUT_PATH          = OUTPUTS_DIR / "phase1_singleturn_classified.csv"

REQUEST_INTERVAL     = 1.0
REGENERATE           = False

import logging
import pandas as pd
from IPython.display import display, Markdown

from src.utils import load_dotenv
from src.providers import GeminiProvider
from src.judge import LLMJudge, CSVBatchClassifier, FewShotExample

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded.")
print(f"Clinician model: {CLINICIAN_MODEL_ID}")
print(f"Judge model:     {JUDGE_MODEL_ID}")

## Few-Shot Examples

One example per CLAMBER uncertainty subclass, adapted to the medical domain.  
These are hand-crafted clinical questions — **not** from the MedQA dataset (no leakage risk).

| Subclass | Label | Pattern |
|---|---|---|
| NK | EPISTEMIC | Model encounters an unfamiliar entity — asks to identify it |
| ICL | EPISTEMIC | Contradictory clinical picture — asks to resolve the contradiction |
| polysemy | ALEATORIC | Medically ambiguous term — multiple valid clinical meanings |
| co-reference | ALEATORIC | Ambiguous pronoun or referent in the patient's description |
| whom | ALEATORIC | Answer depends on patient-specific preference or priority |
| what | ALEATORIC | Underspecified request — multiple valid task interpretations |
| when | ALEATORIC | Ambiguous temporal scope in the description |
| where | ALEATORIC | Ambiguous spatial scope in the description |

In [2]:
FEW_SHOT_EXAMPLES: list[FewShotExample] = [
    # ── EPISTEMIC: NK — model needs to identify an unfamiliar entity ──────
    FewShotExample(
        input="You mentioned having 'MCAS' — could you describe what symptoms it causes for you or what your doctor told you about it?",
        expected_output="EPISTEMIC",
        explanation=(
            "The model doesn't have enough context about this entity to reason "
            "clinically. There is a definite answer — once the patient explains "
            "it, the knowledge gap is fully and permanently resolved."
        ),
    ),
    # ── EPISTEMIC: ICL — contradictory clinical picture to resolve ────────
    FewShotExample(
        input="You described the rash as both 'spreading' and 'fading' — is it currently getting larger or is it clearing up?",
        expected_output="EPISTEMIC",
        explanation=(
            "The two descriptions contradict each other, making clinical "
            "categorisation impossible. There is one correct factual state right "
            "now — the model is resolving a factual contradiction, not a preference."
        ),
    ),
    # ── ALEATORIC: polysemy — term with multiple valid clinical meanings ───
    FewShotExample(
        input="When you say you feel 'weak', do you mean you have no energy and feel fatigued, or that you have actual muscle weakness and difficulty lifting things?",
        expected_output="ALEATORIC",
        explanation=(
            "'Weak' has two clinically distinct meanings (fatigue vs true motor "
            "weakness) that point to completely different differentials. No external "
            "fact can resolve which meaning the patient intends — only the patient can."
        ),
    ),
    # ── ALEATORIC: co-reference — ambiguous referent ──────────────────────
    FewShotExample(
        input="When you said 'it started after the procedure', are you referring to the chest pain or the shortness of breath?",
        expected_output="ALEATORIC",
        explanation=(
            "The pronoun 'it' could validly refer to either symptom. No external "
            "fact can determine which one the patient meant — only the patient's "
            "own context resolves this."
        ),
    ),
    # ── ALEATORIC: whom — depends on patient-specific priority ────────────
    FewShotExample(
        input="Which aspect of your recovery matters most to you — getting back to work quickly, minimising pain, or avoiding surgery?",
        expected_output="ALEATORIC",
        explanation=(
            "The answer depends entirely on this individual patient's values and "
            "priorities. No clinical fact or external knowledge can determine "
            "their personal preference — it is irreducibly patient-specific."
        ),
    ),
    # ── ALEATORIC: what — underspecified request ──────────────────────────
    FewShotExample(
        input="When you ask about treatment options, are you looking for information about medications, surgical approaches, or lifestyle changes?",
        expected_output="ALEATORIC",
        explanation=(
            "The request is underspecified — multiple valid interpretations exist "
            "and the correct path depends entirely on what the patient wants, "
            "not on any clinical fact."
        ),
    ),
    # ── ALEATORIC: when — ambiguous temporal scope ────────────────────────
    FewShotExample(
        input="When you say your symptoms are 'intermittent', do you mean they come and go throughout the day, or that you have symptom-free periods lasting weeks?",
        expected_output="ALEATORIC",
        explanation=(
            "Two valid temporal interpretations exist, each with different clinical "
            "significance. Only the patient knows which pattern applies — "
            "no external fact resolves this."
        ),
    ),
    # ── ALEATORIC: where — ambiguous spatial scope ────────────────────────
    FewShotExample(
        input="When you say the pain is 'everywhere', do you mean it is diffuse throughout your abdomen, or that it shifts between different locations?",
        expected_output="ALEATORIC",
        explanation=(
            "Two spatially distinct clinical patterns (diffuse vs migratory pain) "
            "are both plausible readings. Only the patient can clarify which "
            "pattern they actually experience."
        ),
    ),
]

print(f"Few-shot examples: {len(FEW_SHOT_EXAMPLES)}")
for ex in FEW_SHOT_EXAMPLES:
    print(f"  [{ex.expected_output}] {ex.input[:80]}")

Few-shot examples: 8
  [EPISTEMIC] You mentioned having 'MCAS' — could you describe what symptoms it causes for you
  [EPISTEMIC] You described the rash as both 'spreading' and 'fading' — is it currently gettin
  [ALEATORIC] When you say you feel 'weak', do you mean you have no energy and feel fatigued, 
  [ALEATORIC] When you said 'it started after the procedure', are you referring to the chest p
  [ALEATORIC] Which aspect of your recovery matters most to you — getting back to work quickly
  [ALEATORIC] When you ask about treatment options, are you looking for information about medi
  [ALEATORIC] When you say your symptoms are 'intermittent', do you mean they come and go thro
  [ALEATORIC] When you say the pain is 'everywhere', do you mean it is diffuse throughout your


## Smoke Test

In [ ]:
provider = GeminiProvider(model_id=JUDGE_MODEL_ID)

judge = LLMJudge(
    provider=provider,
    instructions_path=JUDGE_INSTRUCTION,
    few_shot_examples=FEW_SHOT_EXAMPLES,
    label_parser=lambda text: text.strip().upper(),
)

# Smoke test with a question not in few-shot examples
smoke = judge.evaluate("Have you been diagnosed with hypertension or any other heart condition before?")
print(f"Smoke test → label={smoke.label}, error={smoke.error}")
assert smoke.label == "EPISTEMIC", f"Unexpected smoke test label: {smoke.label}"
print("Smoke test passed.")

## Load Phase 1 Results

In [4]:
phase1_df = pd.read_csv(PHASE1_RESULTS_PATH)

# Keep only valid, unblocked rows with a real clarifying question
work_df = phase1_df[
    (~phase1_df["was_blocked"])
    & (phase1_df["clarifying_question"].notna())
    & (phase1_df["clarifying_question"].str.strip() != "")
    & (phase1_df["clarifying_question"] != "BLOCKED")
].copy()

print(f"Phase 1 rows total:  {len(phase1_df)}")
print(f"Usable CQs:          {len(work_df)}")
print()
display(work_df[["id", "ehr_summary", "clarifying_question"]].head(5))

Phase 1 rows total:  100
Usable CQs:          100



,id,ehr_summary,clarifying_question
0,medqa_0982,55-year-old male presenting with: chest pain a...,"What are the findings on chest imaging, such a..."
1,medqa_0799,68-year-old female presenting with: chest pain,What are the patient's current blood pressure ...
2,medqa_1095,45-year-old male presenting with: left-sided a...,What was the specific pathology identified on ...
3,medqa_1228,8-year-old male presenting with: short stature,"Does the patient have any headaches, visual di..."
4,medqa_1054,69-year-old male presenting with: right should...,Does the pain worsen when you lift your arm ou...


## Classify Clarifying Questions

In [5]:
# Save work_df as input CSV for the batch classifier
INPUT_CSV = OUTPUTS_DIR / "phase1_cq_input.csv"
work_df[["id", "clarifying_question"]].to_csv(INPUT_CSV, index=False)
print(f"Input CSV saved: {INPUT_CSV}")

if REGENERATE and OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
    print("Deleted existing output — regenerating.")

classifier = CSVBatchClassifier(
    judge=judge,
    input_csv=INPUT_CSV,
    output_csv=OUTPUT_PATH,
    question_column="clarifying_question",
    id_column="id",
    delay_between_calls=REQUEST_INTERVAL,
)
classifier.run()
print(f"Classification complete → {OUTPUT_PATH}")

01:08:45 [INFO] src.judge — Evaluating: 'What are the findings on chest imaging, such as a chest X-ra...'


01:08:45 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


Input CSV saved: D:\final_project\pilot_study\outputs\medqa\gemini-2.5-flash\phase1_cq_input.csv


01:08:47 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


01:08:47 [INFO] src.judge — label='EPISTEMIC' latency=2.300s


...

01:13:39 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


01:13:41 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


01:13:41 [INFO] src.judge — label='EPISTEMIC' latency=2.237s


01:13:42 [INFO] src.judge — Batch complete — total=100 classified=100 skipped=0 errors=0


Classification complete → D:\final_project\pilot_study\outputs\medqa\gemini-2.5-flash\phase1_singleturn_classified.csv


## Results Summary

In [6]:
classified_df = pd.read_csv(OUTPUT_PATH)

# Drop the empty cq_type placeholder from phase1 before merging to avoid column collision
phase1_for_merge = phase1_df.drop(columns=["cq_type"], errors="ignore")

merged = phase1_for_merge.merge(
    classified_df[["id", "label", "latency_seconds", "error"]].rename(
        columns={"label": "cq_type", "latency_seconds": "judge_latency"}
    ),
    on="id", how="left",
)

valid_labels = {"EPISTEMIC", "ALEATORIC"}
classified = merged[merged["cq_type"].isin(valid_labels)]

print(f"Total classified: {len(classified_df)}")
print(f"Valid labels:     {len(classified)}")
print(f"Invalid/errors:   {len(classified_df) - len(classified)}")
print()
print("Label distribution:")
print(classified["cq_type"].value_counts().to_string())
print()

# Correctness by CQ type
print("Updated correctness by CQ type:")
display(
    classified.groupby("cq_type")[["is_correct_preliminary", "is_correct_updated", "confidence_delta"]]
    .agg({"is_correct_preliminary": "mean", "is_correct_updated": "mean", "confidence_delta": "mean"})
    .round(3)
)

print()
print("Sample classifications:")
display(classified[["id", "cq_type", "clarifying_question", "is_correct_updated", "confidence_delta"]].head(15))

Total classified: 100
Valid labels:     100
Invalid/errors:   0

Label distribution:
cq_type
EPISTEMIC    97
ALEATORIC     3

Updated correctness by CQ type:


,is_correct_preliminary,is_correct_updated,confidence_delta
cq_type,,,
ALEATORIC,0.667,0.667,16.667
EPISTEMIC,0.629,0.742,22.814



Sample classifications:


,id,cq_type,clarifying_question,is_correct_updated,confidence_delta
0,medqa_0982,EPISTEMIC,"What are the findings on chest imaging, such a...",True,30
1,medqa_0799,EPISTEMIC,What are the patient's current blood pressure ...,True,20
2,medqa_1095,EPISTEMIC,What was the specific pathology identified on ...,True,35
3,medqa_1228,EPISTEMIC,"Does the patient have any headaches, visual di...",True,10
4,medqa_1054,EPISTEMIC,Does the pain worsen when you lift your arm ou...,True,30
5,medqa_0215,EPISTEMIC,Are you experiencing any pain or burning with ...,False,20
6,medqa_0155,EPISTEMIC,Has the patient recently taken any medications...,True,30
7,medqa_0886,EPISTEMIC,Are you currently sexually active?,True,20
8,medqa_0640,EPISTEMIC,What are the patient's blood lead levels?,True,5
9,medqa_0004,EPISTEMIC,"Is this a new symptom, or does it happen seaso...",True,5
